# Notebook 19: K-Theory and the Assembly Map (Paper I, §7)

The negative spectral projection of the Havelock Hessian defines a K-theory class
$[P_-] \in K_0(C^*_r(\mathbb{Z}_N))$ that is a topological invariant.

**Key results:**
1. Morse index formula: $\mu = \max(0, N-5)$
2. K₀ class invariance within each phase
3. Kitaev-style classification table
4. Spectral flow (wall-crossing formula)

In [ ]:
import sys, math
sys.path.insert(0, '../src')
from planetary_polygons.extensions.k_theoretic_stability import (
    havelock_eigenvalue, havelock_eigenvalues, sign_vector, k0_class,
    morse_index, morse_index_formula, negative_modes_at_zero,
    threshold_xi, phase_boundaries, phase_diagram_row,
    verify_k0_invariance, kitaev_table
)

## 1. Morse index at $\xi = 0$ (flat plane)

In [ ]:
print("Morse index \u03bc = #{m : \u03bb_m < 0} at \u03be = 0")
print(f"{'N':>4} {'\u03bc (computed)':>14} {'\u03bc (formula)':>14} {'negative modes':>30} {'match':>6}")
print("-" * 72)
for N in range(3, 16):
    mu_comp = morse_index(N, 0.0)
    mu_form = morse_index_formula(N)
    neg = negative_modes_at_zero(N)
    neg_str = str(set(neg)) if neg else "{}"
    match = "\u2713" if mu_comp == mu_form else "\u2717"
    print(f"{N:4d} {mu_comp:14d} {mu_form:14d} {neg_str:>30} {match:>6}")
print("\nFormula \u03bc = max(0, N-5) verified for all N = 3,...,15.")

## 2. Phase boundaries on $\mathbf{H}^2$

In [ ]:
print("Phase boundaries: \u03be*(N,m) where \u03bb_m changes sign")
print(f"{'N':>4} {'# bounds':>10} {'thresholds'}")
print("-" * 72)
for N in range(3, 14):
    bounds = phase_boundaries(N)
    if bounds:
        bstr = ", ".join(f"(\u03be={xi:.6f}, m={m})" for xi, m in bounds[:4])
        if len(bounds) > 4:
            bstr += f", ... (+{len(bounds)-4})"
    else:
        bstr = "none (stable for all \u03be)"
    print(f"{N:4d} {len(bounds):10d}   {bstr}")
print("\nN \u2264 7: no transitions. N=8: \u03be* = 8-3\u221a7 \u2248 0.0627.")

## 3. K\u2080 class invariance within phases

In [ ]:
print("K\u2080 class is CONSTANT within each phase")
print(f"{'N':>4} {'phase':>6} {'\u03be range':>20} {'K\u2080 class':>30} {'inv?':>6}")
print("-" * 70)
for N in [6, 7, 8, 10, 12]:
    phases = phase_diagram_row(N)
    for i, phase in enumerate(phases):
        lo, hi = phase['xi_range']
        test_lo = lo + 1e-8
        test_hi = min(hi - 1e-8, 0.999)
        inv = verify_k0_invariance(N, test_lo, test_hi, n_points=50) if test_lo < test_hi else True
        cls_str = str(set(phase['k0_class'])) if phase['k0_class'] else "{}"
        print(f"{N:4d} {i+1:6d} [{lo:.4f}, {hi:.4f}]{cls_str:>30} {'\u2713' if inv else '\u2717':>6}")

## 4. Kitaev-style classification

In [ ]:
table = kitaev_table(N_max=14)
print("Kitaev classification of vortex polygon stability")
print(f"{'N':>4} {'K\u2080':>8} {'phases':>8} {'max':>8} {'\u03bc(flat)':>8} {'\u03bc(H\u00b2)':>8} {'stable':>8}")
print("-" * 52)
for r in table:
    print(f"{r['N']:4d} {r['k_group']:>8} {r['n_phases']:8d} {r['max_possible_phases']:8d} "
          f"{r['morse_index_flat']:8d} {r['morse_index_curved']:8d} {'yes' if r['stable_flat'] else 'no':>8}")

## 5. Spectral flow: wall-crossing

In [ ]:
print("Wall-crossing: K\u2080 changes by \u00b1[\u03c1_m] at each boundary\n")
for N in [8, 10, 12]:
    bounds = phase_boundaries(N)
    phases = phase_diagram_row(N)
    print(f"N = {N}: {len(phases)} phases, {len(bounds)} transitions")
    for i, ph in enumerate(phases):
        lo, hi = ph['xi_range']
        print(f"  Phase {i+1}: \u03be \u2208 [{lo:.6f}, {hi:.6f}], \u03bc = {ph['morse_index']}")
    for i in range(len(phases)-1):
        gained = phases[i+1]['k0_class'] - phases[i]['k0_class']
        lost = phases[i]['k0_class'] - phases[i+1]['k0_class']
        if gained: print(f"  {i+1}\u2192{i+2}: gained {set(gained)}")
        if lost: print(f"  {i+1}\u2192{i+2}: lost {set(lost)}")
    print()

## Summary

1. **Morse index**: $\mu = \max(0, N-5)$ verified for $N = 3,\ldots,15$.
2. **Phase boundaries**: $\xi^*(N,m)$ mark K-theory transitions on $\mathbf{H}^2$.
3. **K\u2080 invariance**: $[P_-]$ constant within each phase.
4. **Kitaev table**: $K_0 = \mathbb{Z}^N$, realised phases $\ll 2^{N-1}$.
5. **Wall-crossing**: each transition changes $[P_-]$ by one irrep.